# Notebook 1 — Hierarchical Clustering Analysis (HCA)

## What is HCA?

**Hierarchical Clustering Analysis (HCA)** groups data points into a tree of nested clusters without requiring you to specify the number of groups in advance. It works by:

1. Treating each data point as its own cluster.
2. Repeatedly merging the two closest clusters until only one remains.
3. Recording the merge distances to produce a **dendrogram** — a tree diagram.

## Reading a Dendrogram

- The **height** of each horizontal merge line = the distance between the two clusters being joined.
- To choose *k* clusters, draw a horizontal cut line: the number of vertical lines it crosses = number of clusters.
- A **large vertical gap** before a merge suggests the clusters below are genuinely distinct.

## Workshop Narrative

We have survey and performance data for 500 university students. **Without any predefined categories**, can we discover natural *student archetypes* — groups of students who share similar study habits, engagement, and performance patterns?

This unsupervised approach is particularly valuable when you do not yet know what categories exist in your data.

---
*Run each cell in order. All controls are interactive — no code editing required.*

In [ ]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

from scipy.cluster.hierarchy import linkage, fcluster

from utils.data_loader import DataLoaderWidget
from utils.preprocessor import PreprocessorWidget
from utils.plotting import (
    plot_dendrogram,
    plot_cluster_scatter,
    plot_cluster_profile_heatmap,
    plot_correlation_heatmap,
)

print('Libraries loaded ✓')


## Step 1 — Load Data

The sample dataset is pre-loaded. Click **Load CSV** then select your feature columns.
To use your own data, replace the file path with the path to your CSV.

> **Tip:** For clustering, do not select ID columns (`student_id`) or text labels — only numeric/ordinal features.

In [ ]:
# LOAD DATA
loader = DataLoaderWidget(show_label_selector=False)
loader.display()

## Step 2 — Preprocessing

Scaling is important for distance-based methods. **StandardScaler** (zero mean, unit variance) is recommended.

After clicking **Apply Preprocessing**, the scaled data is ready for clustering.

In [ ]:
# HYPERPARAMETERS
preprocessor = PreprocessorWidget(
    source_loader=loader,
    y=None,
    clustering_mode=True,
)
preprocessor.display()


## Step 2b — Feature Scale Check

Before looking at the dendrogram, it is worth checking the **range of values** across your features.

HCA uses Euclidean distances (with Ward linkage) or other distance metrics. If one feature spans
0–2400 and another spans 0–5, the large-scale feature will dominate every pairwise distance
calculation — the algorithm effectively ignores the small-scale features.

The chart below highlights this problem using two deliberately paired columns in the sample dataset:

| Column | Range | Represents |
|---|---|---|
| `avg_weekly_study_hours` | 0–40 | Same behaviour… |
| `total_study_minutes_per_week` | 0–2400 | …just measured in minutes (×60) |
| `lms_logins_per_week` | 0–20 | Same behaviour… |
| `cumulative_lms_sessions_per_semester` | 0–300 | …accumulated over a semester (×13) |

> **Without scaling**, HCA cluster boundaries are almost entirely determined by
> `total_study_minutes_per_week`. The preprocessing step above applies **StandardScaler**
> by default — this is why.

In [ ]:
# OUTPUT
from utils.plotting import plot_feature_scales

_scale_out = widgets.Output()
_scale_btn = widgets.Button(
    description='Show Feature Scale Chart',
    button_style='info',
    icon='bar-chart',
)

def _show_scales(_btn):
    _scale_out.clear_output()
    with _scale_out:
        try:
            df_raw = loader.df
            if df_raw is None:
                display(widgets.HTML('<span style="color:red">Load data first.</span>'))
                return
            fig = plot_feature_scales(df_raw)
            plt.show()
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_scale_btn.on_click(_show_scales)
display(widgets.VBox([_scale_btn, _scale_out]))

## Step 3 — HCA Hyperparameters & Dendrogram

Adjust the controls below to explore different clustering solutions. The dendrogram updates automatically.

| Control | What it does |
|---|---|
| **Linkage method** | How cluster distances are computed. *Ward* minimises within-cluster variance and usually gives the most interpretable result. |
| **Distance metric** | How individual sample distances are measured. Ward requires Euclidean. |
| **Colour threshold** | The horizontal cut height. Branches below this height are coloured differently to show cluster membership. |
| **Number of clusters** | The number of groups to extract from the tree. |
| **Truncate at p** | Show only the last *p* merges to keep the dendrogram readable for large datasets. |

> **Note:** For datasets with > 1,000 rows, computing the full dendrogram can be slow. A sampling option is provided.

In [ ]:
# HYPERPARAMETERS
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

_hca_out = widgets.Output()
_cluster_out = widgets.Output()

# Sampling controls for large datasets
_sample_check = widgets.Checkbox(
    value=False,
    description='Sample 500 rows for dendrogram (recommended if n > 1000)',
    layout=widgets.Layout(width='480px'),
)

# Linkage & metric
_linkage_dd = widgets.Dropdown(
    options=['ward', 'complete', 'average', 'single'],
    value='ward',
    description='Linkage:',
    style={'description_width': '120px'},
)
_metric_dd = widgets.Dropdown(
    options=['euclidean', 'cosine', 'manhattan'],
    value='euclidean',
    description='Distance metric:',
    style={'description_width': '120px'},
)
_metric_note = widgets.HTML(
    '<span style="color:#888; font-size:0.85em">ℹ Ward linkage requires euclidean metric.</span>'
)

def _update_metric_state(change):
    if _linkage_dd.value == 'ward':
        _metric_dd.value = 'euclidean'
        _metric_dd.disabled = True
    else:
        _metric_dd.disabled = False

_linkage_dd.observe(_update_metric_state, names='value')
_update_metric_state(None)

# Threshold & cluster count
_threshold_slider = widgets.FloatSlider(
    value=10.0, min=0.0, max=50.0, step=0.5,
    description='Colour threshold:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
    readout_format='.1f',
)
_n_clusters_slider = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description='N clusters:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
)
_truncate_slider = widgets.IntSlider(
    value=30, min=5, max=50, step=1,
    description='Truncate at p:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
)

_run_btn = widgets.Button(
    description='Compute & Plot',
    button_style='primary',
    icon='bar-chart',
)

_Z = None  # linkage matrix cache
_X_used = None

def _run_hca(_btn):
    global _Z, _X_used
    _hca_out.clear_output()
    _cluster_out.clear_output()

    with _hca_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            n_rows = len(X_scaled)
            if n_rows > 1000:
                display(widgets.HTML(
                    f'<span style="color:orange">⚠ Dataset has {n_rows} rows. '
                    'Dendrogram computation may be slow. Consider enabling sampling above.</span>'
                ))

            X_use = X_scaled
            if _sample_check.value and n_rows > 500:
                X_use = X_scaled.sample(500, random_state=42)
                display(widgets.HTML('<span style="color:blue">ℹ Using 500-row sample for dendrogram.</span>'))

            _X_used = X_use
            method = _linkage_dd.value
            metric = _metric_dd.value if method != 'ward' else 'euclidean'

            display(widgets.HTML('Computing linkage matrix...'))
            _Z = linkage(X_use.values, method=method, metric=metric)

            # Adjust threshold slider max to actual data range
            max_dist = float(_Z[:, 2].max())
            _threshold_slider.max = round(max_dist, 1)
            if _threshold_slider.value > max_dist:
                _threshold_slider.value = round(max_dist * 0.3, 1)

            fig = plot_dendrogram(
                _Z,
                truncate_p=_truncate_slider.value,
                color_threshold=_threshold_slider.value,
                title=f'Dendrogram — {method} linkage, {metric} distance',
            )
            plt.show()

            # Cluster assignment
            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')
            counts = pd.Series(labels).value_counts().sort_index()
            display(widgets.HTML(f'<b>Cluster sizes ({n_clust} clusters):</b>'))
            display(counts.rename('count').to_frame())

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

def _update_dendrogram(change):
    if _Z is None:
        return
    _hca_out.clear_output()
    with _hca_out:
        try:
            fig = plot_dendrogram(
                _Z,
                truncate_p=_truncate_slider.value,
                color_threshold=_threshold_slider.value,
                title=f'Dendrogram — {_linkage_dd.value} linkage',
            )
            plt.show()
            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')
            counts = pd.Series(labels).value_counts().sort_index()
            display(widgets.HTML(f'<b>Cluster sizes ({n_clust} clusters):</b>'))
            display(counts.rename('count').to_frame())
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_threshold_slider.observe(_update_dendrogram, names='value')
_n_clusters_slider.observe(_update_dendrogram, names='value')
_truncate_slider.observe(_update_dendrogram, names='value')
_run_btn.on_click(_run_hca)

display(widgets.VBox([
    widgets.HTML('<h3>🌳 Step 3 — HCA Controls</h3>'),
    _sample_check,
    _linkage_dd,
    _metric_dd,
    _metric_note,
    _threshold_slider,
    _n_clusters_slider,
    _truncate_slider,
    _run_btn,
]))
display(_hca_out)

## Step 4 — Cluster Profiles

After choosing your number of clusters, run this cell to see what each cluster looks like in terms of:
- **Cluster sizes** — how many students in each group
- **Feature means per cluster** — what distinguishes each group
- **2D scatter plot** — students projected into 2 dimensions (PCA) and coloured by cluster

In [ ]:
# OUTPUT
_profile_out = widgets.Output()
_profile_btn = widgets.Button(
    description='Show Cluster Profiles',
    button_style='info',
    icon='table',
)

def _show_profiles(_btn):
    _profile_out.clear_output()
    with _profile_out:
        try:
            if _Z is None or _X_used is None:
                display(widgets.HTML('<span style="color:red">Run Step 3 first.</span>'))
                return

            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')

            # Cluster size table
            counts = pd.Series(labels, name='cluster').value_counts().sort_index()
            display(widgets.HTML('<b>Cluster sizes:</b>'))
            display(counts.rename('n_students').to_frame())

            # Feature means heatmap (on original scale if possible)
            X_orig = loader.X_df.copy() if loader.X_df is not None else _X_used
            X_orig = X_orig.select_dtypes(include='number').loc[_X_used.index]
            cluster_means = X_orig.groupby(labels).mean()
            cluster_means.index = [f'Cluster {i}' for i in cluster_means.index]

            display(widgets.HTML('<b>Mean feature values per cluster:</b>'))
            fig = plot_cluster_profile_heatmap(cluster_means)
            plt.show()

            # 2D scatter
            fig2 = plot_cluster_scatter(
                _X_used,
                labels,
                method='PCA',
                title=f'Student Clusters — PCA projection ({n_clust} clusters)',
            )
            plt.show()

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_profile_btn.on_click(_show_profiles)
display(widgets.VBox([_profile_btn, _profile_out]))

## Step 5 — Interpretation

Look at the cluster profiles above and complete the descriptions below. There are no right or wrong answers — this is about developing your interpretation.

---

**Cluster 1** appears to represent students who...

*(e.g., "show high tutorial attendance and study hours but moderate exam scores — possibly anxious high-effort students")*

---

**Cluster 2** appears to represent students who...

---

**Cluster 3** appears to represent students who...

---

**Cluster 4** appears to represent students who...

---

### Reflection Questions

1. **How many clusters did you choose, and why?** (Look for the largest vertical gap in the dendrogram.)
2. **Which features most differentiated the clusters?** (Bright vs. dark cells in the heatmap.)
3. **Are any clusters very small?** This may indicate outlier students — or it may be meaningful.
4. **Does the 2D scatter plot match the dendrogram groupings?** If not, why might that be?

> **Next step:** Open Notebook 2 to formalise these groupings with K-Means clustering.